In [7]:
import pandas as pd

data = pd.read_csv(
    # "s3://ml-bucket-demo-22/cleaned-data/part-00000-e992e629-1d3a-4f99-9675-b2ad16b51f30-c000.csv"
    "s3://ml-bucket-demo-22/cleaned-data/part-00000-e992e629-1d3a-4f99-9675-b2ad16b51f30-c000.csv"
)

print(data.head())

   house_size_sqft  bedrooms  age_years   price
0             2700         5          4  390000
1             3000         5          3  420000
2              800         2         20  150000
3             1200         3         18  200000
4             1500         3         10  250000


In [8]:
# Prepare features and target

X = data[['house_size_sqft','bedrooms','age_years']]

y = data['price']

In [9]:
# Split the dataset

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [10]:
# Train Model
from sklearn.linear_model import LinearRegression

lr_model = LinearRegression()

lr_model.fit(X_train, y_train)

print("Model trained successfully")

Model trained successfully


In [12]:
# Evaluate Model

from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

pred = lr_model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, pred))
print("RMSE:", rmse)

print("R2 score:", r2_score(y_test, pred))

RMSE: 721.5860656027367
R2 score: 0.999893737459169


In [13]:
# Save the model
import joblib

joblib.dump(lr_model,"model.joblib")

print("Model saved")

Model saved


In [14]:
# Create tar file
import tarfile

with tarfile.open("model.tar.gz","w:gz") as tar:
    tar.add("model.joblib")

print("tar file created")

tar file created


In [15]:
# Upload Model To S3

import sagemaker

session = sagemaker.Session()

bucket = session.default_bucket()

model_s3_path = session.upload_data(
    path="model.tar.gz",
    bucket=bucket,
    key_prefix="model"
)

print(model_s3_path)

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
s3://sagemaker-ap-southeast-2-480827623277/model/model.tar.gz


In [18]:
# Deploy End Point
from sagemaker.sklearn.model import SKLearnModel

# Sagemaker 

role = "arn:aws:iam::480827623277:role/ml-sagemaker-role"

sm_model = SKLearnModel(
    model_data=model_s3_path,
    role=role,
    entry_point="script.py",
    framework_version="1.0-1"
)

predictor = sm_model.deploy(
    instance_type="ml.t2.medium",
    initial_instance_count=1
)

print("Endpoint created")

------!Endpoint created


In [ ]:
predictor.predict([[2000,4,5]])

In [ ]:
predictor.delete_endpoint()